# Step 2 — Ilastik masks, image registration, and manual-review workflow

This notebook coordinates Step2 processing and explains the optional Step2/Step3 manual review steps. Actual selection and numerical parameters are read from the **user's YAML configuration**.

Processing sequence:

`manifest_recordings.csv` + Ilastik probability H5 files → automatic Step2 masks and registration → optional Step2 manual bregma review → registration rerun → Step3 anatomical QC/annotation → optional targeted Step2/Step3 reruns → optional final Step3 manual registration.

- Step1–Step3 images and masks use **`rotated_fullres`** space; Step4 raw Zarr uses **`rotated_scaled`** space.
- Set `CONFIG_PATH` in the code cells below. It is recommended to store the working `config.yaml` in `project.analysis_parent_dir` for reproducibility.
- Step2 uses the Step1 manifest, `analysis_group`, Step2 settings, and exclusions; it does **not** use `analysis_selection` as its primary recording filter.
- If `step2.qc.apply_thresholds` is absent, the code defaults to `False`; configured numeric QC thresholds then do not create threshold-based failure reasons, although structural failures may still mark records as bad.
- Missing inputs are reported in `step2_missing_inputs.csv`. Step3 may write a separate `step3_missing_inputs.csv`.

**Run-mode note:** the Step2 batch cell performs processing. Later cells print commands for standalone Napari tools rather than launching them within Jupyter.

## 0. Imports and source directory

The notebook imports Python modules by their standard filenames. Set `SOURCE_DIR` to the directory containing `common.py` and the Step2/Step3 modules. The default path in the next cell assumes a particular working directory; adjust it for your checkout.

In [ ]:

import sys
from pathlib import Path

# Expected project layout:
# project/notebook/Step2_MasksCreation.ipynb
# project/src/common.py
# project/src/step2_masks_registration.py

# The Jupyter working directory must be project/notebook/.
NOTEBOOK_DIR = Path.cwd().resolve()
SRC_DIR = (NOTEBOOK_DIR.parent / "src").resolve()

required_files = ("common.py", "step2_masks_registration.py")
missing_files = [
    name for name in required_files
    if not (SRC_DIR / name).is_file()
]

if missing_files:
    raise FileNotFoundError(
        f"Could not find the required source files: {', '.join(missing_files)}. "
        f"Expected location: {SRC_DIR}. "
        "Run this notebook from the project's notebook/ directory."
    )

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print('Source folder:', SRC_DIR)

# Preserve compatibility with the existing CLI command cell.
SOURCE_DIR = SRC_DIR

import pandas as pd
from IPython.display import display

from common import (
    load_config,
    create_or_load_project,
    build_standard_paths,
    get_stem_keys,
    get_path_resolving_config,
)

from step2_masks_registration import (
    get_analysis_group_config,
    build_step2_index_from_manifest,
    choose_reference_row_for_group,
    run_step2,
    read_probs_h5,
    get_step2_mask_wavelengths,
    get_registration_wavelength,
    build_brain_and_ring_from_sources,
    bregma_centroid_from_sources,
    build_bad_mask_from_sources,
    combine_source_masks,
)


## 1. Load the configuration and existing project paths

Set `CONFIG_PATH` to the **same YAML configuration file used in Step 1**. The Windows path shown in the code is only an example and must be replaced with the location of your own configuration file.

After Step 1, the configuration should already contain the analysis directory path:

```yaml
project:
  directory_to_save_main_analysis: "F:/Analysis/My_experiment_yyyy-mm-dd"
```

The notebook uses:

- `load_config()` to read the existing YAML configuration.
- `create_or_load_project()` to locate the analysis directory and ensure the required output subdirectories exist.
- `build_standard_paths()` to retrieve the paths to the Step 1 manifest, Step 2 metadata, masks, and other pipeline outputs.

When `directory_to_save_main_analysis` is set, the existing analysis directory is used regardless of `analysis_parent_dir`, `analysis_folder_name`, or `create_new_analysis_folder`. In this case, `create_or_load_project()` does not modify the YAML configuration or create a new configuration snapshot.

**Important:** Do not reset `directory_to_save_main_analysis` to `null` when continuing an existing analysis. Otherwise, the function will enter the project-creation branch, which may raise `FileExistsError` if the target directory is already populated.

Step 2 requires the `manifest_recordings.csv` generated in Step 1 and the corresponding Ilastik probability maps.

For scientific reproducibility, retain the final configuration together with the analysis results. The initial configuration snapshot is not automatically updated when processing parameters change.

In [ ]:
# Example path from the original analysis; replace with your own YAML location.
CONFIG_PATH = Path(r"config.yaml")

config = load_config(CONFIG_PATH)
stem_keys = get_stem_keys(config)
analysis_group = get_analysis_group_config(config)
project_paths = create_or_load_project(config, CONFIG_PATH)
paths = build_standard_paths(config, project_paths)
path_cfg = get_path_resolving_config(config, "step2")

print("CONFIG_PATH:", CONFIG_PATH)
print("analysis_root:", project_paths["analysis_root"])
print("stem_keys:", stem_keys)
print("analysis_group:", analysis_group)
print("path_resolving:", path_cfg)
print("manifest_path:", paths["manifest_path"])
print("metadata_registration_csv:", paths["metadata_registration_csv"])
print("masks_root:", paths["masks_root"])


## 2. YAML settings used by Step2 and subsequent Step3 review

Actual values come from the user's YAML file, not from these examples:

- `analysis_group.group_by`: registration grouping keys.
- `analysis_group.reference.match` and `analysis_group.reference.variant_keys`: reference selection, including possible fallback with `ref_exact_match=False`.
- `step2.mask_sources`: wavelengths, Ilastik class indices, thresholds, dilation and semantic roles.
- `step2.registration`: image wavelength, feature method, ROI mode, anchoring and search settings.
- `step2.qc.apply_thresholds`: optional Boolean, **defaults to `False` if omitted**. Structural invalidity is still checked.
- `step3.review`, `step3.annotation`, `step3.registration_qc`: controls for subsequent manual QC and anatomical labeling.
- `path_resolving`: global root/drive remapping, when needed.

Step1 discovery uses `discovery.metadata_filters`; `analysis_selection` does not directly select records for Step2. Do not assume example wavelengths, groups, or animals apply to a different experiment.

## 3. Optional: inspect one Ilastik probability H5 file

Use these helper functions to inspect HDF5 channels and the masks selected by `step2.mask_sources`. This is optional and does not run Step2 registration.

In [ ]:
# Functions
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

def infer_wavelength_from_path(h5_path):
    """Infer a wavelength label from a numeric parent directory.

    Parameters
    ----------
    h5_path : str or pathlib.Path
        Path to a probability H5 file.

    Returns
    -------
    str or None
        Nearest all-digit path component, or None if none exists.

    Notes
    -----
        Example path: ``.../ilastik_reference_images/505/file_Probabilities.h5``.
        The wavelength is inferred from the folder name, not the H5 content.
    """
    p = Path(h5_path)
    for part in reversed(p.parts):
        text = str(part).strip()
        if text.isdigit():
            return text
    return None


def load_ilastik_probabilities(h5_path, dataset_name="exported_data"):
    """Inspect and load an Ilastik probability dataset.

    Parameters
    ----------
    h5_path : str or pathlib.Path
        Path to an Ilastik probability H5 file.
    dataset_name : str, default='exported_data'
        Dataset key; the pipeline normally obtains it from ``step2.h5.dataset``.

    Returns
    -------
    numpy.ndarray
        Probability map with shape ``(y, x, class)``. A singleton leading axis
        is removed if the exported array has shape ``(1, y, x, class)``.

    Notes
    -----
        Displays H5 dataset names and shapes for optional manual inspection.
    """
    h5_path = Path(h5_path)

    with h5py.File(h5_path, "r") as f:
        print("H5 structure:")

        def print_structure(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(" ", name, obj.shape, obj.dtype)

        f.visititems(print_structure)

        if dataset_name not in f:
            raise KeyError(
                f"Dataset {dataset_name!r} not found. "
                f"Available top-level keys: {list(f.keys())}"
            )

        probs = f[dataset_name][:]

    if probs.ndim == 4 and probs.shape[0] == 1:
        probs = probs[0]

    if probs.ndim != 3:
        raise ValueError(
            f"Expected probabilities with shape (y, x, classes), "
            f"got {probs.shape}"
        )

    return probs


def normalise_mask_sources(config):
    """Flatten configured semantic mask sources into a display table.

    Parameters
    ----------
    config : dict
        Project YAML configuration containing ``step2.mask_sources``.

    Returns
    -------
    pandas.DataFrame
        One row per enabled source, with role, wavelength, class index,
        threshold, required flag, bad-mask flag and dilation.

    Notes
    -----
        Both single-source and ``sources`` list styles are supported. For
        example only, the brain and glare roles may use different wavelengths;
        actual channel indices and thresholds are read from YAML.
    """
    step2_cfg = (config or {}).get("step2", {}) or {}
    mask_sources = step2_cfg.get("mask_sources", {}) or {}

    rows = []

    for role, role_cfg in mask_sources.items():
        if role_cfg is None:
            continue

        role_cfg = dict(role_cfg)
        if role_cfg.get("enabled", True) is False:
            continue

        role_required = bool(role_cfg.get("required", False))
        role_add_to_bad = bool(role_cfg.get("add_to_bad_mask", False))

        if "sources" in role_cfg and role_cfg["sources"] is not None:
            sources = role_cfg["sources"]
        else:
            sources = [role_cfg]

        for i, src in enumerate(sources):
            if src is None:
                continue

            src = dict(src)
            if src.get("enabled", True) is False:
                continue

            wavelength = src.get("wavelength", role_cfg.get("wavelength"))
            class_index = src.get("class_index", role_cfg.get("class_index"))
            threshold = src.get("threshold", role_cfg.get("threshold"))

            if wavelength is None or class_index is None:
                continue

            rows.append({
                "role": str(role),
                "source_i": int(i),
                "wavelength": str(wavelength),
                "class_index": int(class_index),
                "threshold": None if threshold is None else float(threshold),
                "required": bool(src.get("required", role_required)),
                "add_to_bad_mask": bool(src.get("add_to_bad_mask", role_add_to_bad)),
                "dilate": int(src.get("dilate", role_cfg.get("dilate", 0)) or 0),
            })

    return pd.DataFrame(rows)


def get_sources_for_wavelength_and_class(df_sources, wavelength, class_index):
    """Select mask-source definitions for one wavelength and class index.

    Parameters
    ----------
    df_sources : pandas.DataFrame
        Flattened source table.
    wavelength : str
        Wavelength name.
    class_index : int
        Zero-based class index.

    Returns
    -------
    pandas.DataFrame
        Matching source definitions.
    """
    if df_sources.empty:
        return df_sources

    return df_sources[
        (df_sources["wavelength"].astype(str) == str(wavelength)) &
        (df_sources["class_index"].astype(int) == int(class_index))
    ].copy()


def check_required_sources(df_sources, h5_wavelength, n_classes):
    """Report configured class indices outside one H5 wavelength's channels.

    Parameters
    ----------
    df_sources : pandas.DataFrame
        Flattened mask-source table.
    h5_wavelength : str
        Wavelength associated with the inspected file.
    n_classes : int
        Number of probability channels in this H5 dataset.

    Notes
    -----
        Checks only sources assigned to the inspected wavelength, not every
        source in the project configuration.
    """
    if df_sources.empty:
        print("No step2.mask_sources found in config.")
        return

    current = df_sources[df_sources["wavelength"].astype(str) == str(h5_wavelength)].copy()

    if current.empty:
        print(f"No mask_sources configured for wavelength={h5_wavelength!r}.")
        return

    for _, r in current.iterrows():
        class_idx = int(r["class_index"])
        role = r["role"]

        if class_idx >= n_classes:
            status = "ERROR" if r["required"] else "WARNING"
            print(
                f"[{status}] role={role!r}, wavelength={h5_wavelength}, "
                f"class_index={class_idx} is outside available classes 0..{n_classes - 1}"
            )


def plot_ilastik_classes_with_mask_sources(
    h5_path,
    config=None,
    wavelength=None,
    dataset_name=None,
    cmap_prob="viridis",
    cmap_mask="gray",
):
    """Visualize Ilastik class probabilities and configured threshold masks.

    Parameters
    ----------
    h5_path : str or pathlib.Path
        Probability H5 file to inspect.
    config : dict or None, optional
        Configuration providing ``step2.mask_sources`` and ``step2.h5.dataset``.
    wavelength : str or None, optional
        Wavelength label; if missing, infer it from the path.
    dataset_name : str or None, optional
        H5 dataset name; defaults to configured value or ``exported_data``.
    cmap_prob, cmap_mask : str
        Colormap names for probabilities and binary masks.

    Notes
    -----
        The first plot row shows each available class probability map. The
        second row shows binary masks for class indices configured at the
        current wavelength. Multiple semantic roles can refer to the same
        wavelength or class.
    """
    h5_path = Path(h5_path)

    if dataset_name is None:
        dataset_name = (
            (config or {})
            .get("step2", {})
            .get("h5", {})
            .get("dataset", "exported_data")
        )

    if wavelength is None:
        wavelength = infer_wavelength_from_path(h5_path)

    if wavelength is None:
        raise ValueError(
            "Could not infer wavelength from path. "
            "Pass wavelength manually, e.g. wavelength='505'."
        )

    probs = load_ilastik_probabilities(h5_path, dataset_name=dataset_name)
    y_size, x_size, n_classes = probs.shape

    df_sources = normalise_mask_sources(config)

    print()
    print(f"Loaded probabilities: {probs.shape}")
    print(f"Wavelength: {wavelength}")
    print(f"Dataset: {dataset_name}")
    print()

    print("Configured mask_sources:")
    if df_sources.empty:
        print("  No step2.mask_sources found.")
    else:
        try:
            display(df_sources)
        except NameError:
            print(df_sources)

    print()
    check_required_sources(df_sources, wavelength, n_classes)

    n_rows = 2
    n_cols = n_classes

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(4 * n_cols, 8),
        squeeze=False,
    )

    used_masks = []
    bad_masks = []

    for class_idx in range(n_classes):
        prob = probs[..., class_idx].astype(np.float32)
        src_rows = get_sources_for_wavelength_and_class(
            df_sources=df_sources,
            wavelength=wavelength,
            class_index=class_idx,
        )

        if src_rows.empty:
            title_name = "not used in mask_sources"
            threshold_text = "no threshold"
            mask = None
        else:
            names = []
            masks_for_class = []

            for _, src in src_rows.iterrows():
                role = src["role"]
                thr = src["threshold"]
                required = bool(src["required"])
                add_to_bad = bool(src["add_to_bad_mask"])
                dilate = int(src["dilate"])

                name = (
                    f"{role}"
                    f"\nthr={thr}"
                    f"\nrequired={required}"
                    f"\nadd_to_bad={add_to_bad}"
                    f"\ndilate={dilate}"
                )
                names.append(name)

                if thr is not None:
                    m = prob > float(thr)
                    masks_for_class.append((role, m, add_to_bad))
                    used_masks.append((role, class_idx, m, add_to_bad))

                    if add_to_bad:
                        bad_masks.append(m)

            title_name = "\n---\n".join(names)
            threshold_text = "configured"

            if masks_for_class:
                mask = np.zeros_like(prob, dtype=bool)
                for _, m, _ in masks_for_class:
                    mask |= m
            else:
                mask = None

        # -----------------------------
        # Probability map
        # -----------------------------
        ax = axes[0, class_idx]
        im = ax.imshow(prob, cmap=cmap_prob, vmin=0, vmax=1)
        ax.set_title(
            f"Probability\n"
            f"class_index={class_idx}\n"
            f"{title_name}",
            fontsize=9,
        )
        ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        # -----------------------------
        # Thresholded mask
        # -----------------------------
        ax = axes[1, class_idx]

        if mask is not None:
            ax.imshow(mask, cmap=cmap_mask, vmin=0, vmax=1)
            ax.set_title(
                f"Mask\n"
                f"class_index={class_idx}\n"
                f"{threshold_text}",
                fontsize=9,
            )
        else:
            ax.imshow(np.zeros_like(prob), cmap=cmap_mask, vmin=0, vmax=1)
            ax.set_title(
                f"No mask source\n"
                f"class_index={class_idx}",
                fontsize=9,
            )

        ax.axis("off")

    fig.suptitle(
        f"Ilastik mask_sources check\n"
        f"{h5_path.name}\n"
        f"wavelength={wavelength}",
        fontsize=14,
    )
    fig.tight_layout()
    plt.show()

    # -----------------------------
    # Optional combined bad mask preview
    # -----------------------------
    if bad_masks:
        combined_bad = np.zeros_like(bad_masks[0], dtype=bool)
        for m in bad_masks:
            combined_bad |= m.astype(bool)

        plt.figure(figsize=(5, 5))
        plt.imshow(combined_bad, cmap="gray", vmin=0, vmax=1)
        plt.title(
            f"Combined bad mask preview\n"
            f"from add_to_bad_mask=True sources\n"
            f"wavelength={wavelength}"
        )
        plt.axis("off")
        plt.show()
    else:
        combined_bad = None
        print(f"No add_to_bad_mask=True sources for wavelength={wavelength}.")

    return probs, df_sources, combined_bad

In [ ]:
# Optional H5 preview. Example path: replace with your own Step1 H5 export.
h5_path = r"F:\PT2021_demonstration_analysis\Main_analysis_2026-11-21\out\step1_project\ilastik_reference_images\505\m18_1d-PT_Awake_Rest-state_Probabilities.h5"  # Example only: another wavelength and stem-specific Ilastik H5 path.

if h5_path is not None:
    probs, df_sources, combined_bad = plot_ilastik_classes_with_mask_sources(
        h5_path, config=config, wavelength=None,
    )
else:
    print("Ilastik H5 preview skipped (h5_path=None).")


## 4. Validate Step1 inputs before the full Step2 run

The next cell checks the Step1 manifest against the configured reference BMP and Ilastik probability H5 requirements. Pass the **complete** configuration to `build_step2_index_from_manifest`, not just `config['step2']`: the index also needs `metadata.stem_keys`, `analysis_group` and `variant_keys`.

Missing or invalid input rows are reported by Step2 in `step2_missing_inputs.csv`.

In [ ]:
df_index = build_step2_index_from_manifest(
    manifest_path=paths["manifest_path"],
    cfg=config,   # Pass the full config, not config["step2"]
    paths=paths,
    require_h5=True,
    from_drive=path_cfg.get("from_drive"),
    to_drive=path_cfg.get("to_drive"),
)

print("valid rows:", len(df_index))

if df_index.empty:
    missing_path = paths["step2_root"] / "step2_missing_inputs.csv"
    print("No valid Step2 rows.")
    print("Check missing inputs:", missing_path)
else:
    show_cols = ["stem", "group_id"] + [k for k in stem_keys if k in df_index.columns]
    path_cols = sorted([c for c in df_index.columns if c.startswith(("ref_", "prob_")) and c.endswith("_exists")])
    variant_cols = [c for c in df_index.columns if c.endswith("_base") or c.endswith("_repeat")]
    existing_cols = [c for c in show_cols + path_cols + variant_cols if c in df_index.columns]
    display(df_index[existing_cols].head(20))

missing_path = paths["step2_root"] / "step2_missing_inputs.csv"
if missing_path.exists():
    missing_df = pd.read_csv(missing_path)
    if not missing_df.empty:
        print(f"[WARN] Excluded {len(missing_df)} recording(s); see {missing_path}")
        display(missing_df[[c for c in ["group_id", "stem", "error"] if c in missing_df.columns]])


## 4b. Preview masks before full Step2 registration

This optional section **does not write `metadata_registration.csv`**. It displays semantic masks for a small sample of stems, based on the current YAML thresholds and morphology settings (`step2.mask_sources`, `step2.morphology`). Depending on `step2.preview` and the notebook preview controls, it **may save PNG previews and `mask_preview_stats.csv`**.

Suggested workflow:

1. Update mask-source and morphology values in your YAML.
2. Re-run the configuration and Step2-index cells.
3. Run the preview cell and inspect the resulting brain, bregma, bad, ring, vessel and background masks.
4. Run the full Step2 cell only after reviewing the preview.

A targeted Step2 rerun may merge recomputed groups into an existing `metadata_registration.csv`. It does not overwrite unrelated rows when the supported partial-rerun mode is used.

In [ ]:
# Preview overrides: None uses step2.preview or the initial groups in df_index.
PREVIEW_GROUPS = None
PREVIEW_MAX_GROUPS = None
PREVIEW_MAX_STEMS_PER_GROUP = None
PREVIEW_SAVE_PNG = None
PREVIEW_SHOW_INLINE = None

import numpy as np
import matplotlib.pyplot as plt
import imageio.v3 as iio
from pathlib import Path


def _preview_cfg_value(name, default):
    """Read one Step2 preview setting from YAML with a notebook fallback."""
    preview_cfg = ((config.get("step2", {}) or {}).get("preview", {}) or {})
    return preview_cfg.get(name, default)


def _as_list_or_none(x):
    if x is None:
        return None
    if isinstance(x, str):
        return [x]
    return [str(v) for v in x]


def _select_preview_groups(df_index):
    """Choose groups for mask preview using notebook and YAML selections."""
    groups = _as_list_or_none(PREVIEW_GROUPS)
    if groups is None:
        groups = _as_list_or_none(_preview_cfg_value("groups", None))
    max_groups = PREVIEW_MAX_GROUPS if PREVIEW_MAX_GROUPS is not None else int(_preview_cfg_value("max_groups", 2))
    available = sorted(df_index["group_id"].dropna().astype(str).unique().tolist())
    if groups:
        selected = [g for g in groups if g in set(available)]
    else:
        selected = available[:max_groups]
    return selected


def _read_gray_preview(path_value):
    """Read a preview image as two-dimensional float32 data, or return None."""
    p = Path(str(path_value))
    if not p.exists():
        return None
    arr = iio.imread(p)
    if arr.ndim == 3:
        arr = arr[..., :3].mean(axis=-1)
    return arr.astype(np.float32)


def _norm_img(img, p=(1, 99)):
    """Normalize finite image intensities to [0, 1] for display."""
    vals = img[np.isfinite(img)]
    if vals.size < 10:
        return np.zeros_like(img, dtype=np.float32)
    lo, hi = np.percentile(vals, p)
    out = (img - lo) / (hi - lo + 1e-6)
    return np.clip(out, 0, 1)


def _overlay_mask(ax, base, mask, title, alpha=0.5):
    """Draw a semi-transparent spatial mask over a grayscale image."""
    ax.imshow(_norm_img(base), cmap="gray")
    overlay = np.ma.masked_where(~mask.astype(bool), mask.astype(bool))
    ax.imshow(overlay, alpha=alpha)
    ax.set_title(f"{title}\nN={int(mask.sum())}")
    ax.axis("off")


def preview_step2_masks_for_row(row, *, save_dir: Path | None = None, show: bool = True):
    """Preview probability-derived masks for a single Step2 recording.

    Parameters
    ----------
    row : pandas.Series
        Indexed recording with configured probability and reference image paths.
    save_dir : pathlib.Path or None, optional
        Directory for PNG previews.
    show : bool, default=True
        Whether to display plots inline.

    Returns
    -------
    dict
        Pixel counts, brain/bad fraction, bregma status and recording identity.

    Notes
    -----
        This visualization uses the current YAML mask sources and morphology
        settings but does not write ``metadata_registration.csv``. The optional
        PNG output is a notebook preview, not the registered Step2 result.
    """
    stem = str(row["stem"])
    step2_cfg = config.get("step2", {}) or {}
    morph = step2_cfg.get("morphology", {}) or {}
    h5_dataset = step2_cfg.get("h5", {}).get("dataset", "exported_data")
    registration_wavelength = get_registration_wavelength(config)
    mask_wavelengths = get_step2_mask_wavelengths(config)

    probs_by_wave = {}
    for wl in mask_wavelengths:
        h5_path = row.get(f"h5_{wl}", "")
        if isinstance(h5_path, str) and h5_path.strip() and Path(h5_path).exists():
            probs_by_wave[str(wl)] = read_probs_h5(Path(h5_path), dataset=h5_dataset)

    brain, ring = build_brain_and_ring_from_sources(
        probs_by_wave=probs_by_wave,
        config_or_step2=config,
        close_r=int(morph.get("close_r_brain", 3)),
        r_ring=int(morph.get("ring_radius", 10)),
    )
    cy, cx, bregma_mask, bregma_found = bregma_centroid_from_sources(probs_by_wave, config)
    bad = build_bad_mask_from_sources(probs_by_wave, config, brain.shape)
    vessels = combine_source_masks(probs_by_wave, config, "vessels", required=False)
    if vessels is None:
        vessels = np.zeros_like(brain, dtype=bool)
    background = (~brain.astype(bool)) & (~bad.astype(bool)) & (~bregma_mask.astype(bool))

    img = _read_gray_preview(row.get(f"bmp_{registration_wavelength}", row.get(f"ref_{registration_wavelength}_path", "")))
    if img is None:
        # fallback to first available ref image
        for c in [c for c in row.index if str(c).startswith("bmp_") or (str(c).startswith("ref_") and str(c).endswith("_path"))]:
            img = _read_gray_preview(row.get(c, ""))
            if img is not None:
                break
    if img is None:
        img = np.zeros_like(brain, dtype=np.float32)

    fig, axes = plt.subplots(1, 7, figsize=(24, 4))
    axes[0].imshow(_norm_img(img), cmap="gray")
    axes[0].set_title(f"{stem}\nimage {registration_wavelength}")
    axes[0].axis("off")
    _overlay_mask(axes[1], img, brain, "brain")
    _overlay_mask(axes[2], img, bad, "bad")
    _overlay_mask(axes[3], img, ring, "ring")
    _overlay_mask(axes[4], img, bregma_mask, f"bregma found={bregma_found}")
    _overlay_mask(axes[5], img, vessels, "vessels")
    _overlay_mask(axes[6], img, background, "background")
    if bregma_found and cy is not None and cx is not None:
        axes[0].plot([cx], [cy], marker="x", markersize=8)
        axes[4].plot([cx], [cy], marker="x", markersize=8)
    fig.tight_layout()

    stats = {
        "stem": stem,
        "group_id": row.get("group_id", ""),
        "brain_area": int(brain.sum()),
        "bad_area": int(bad.sum()),
        "bad_fraction_of_brain": float(bad.sum() / max(brain.sum(), 1)),
        "ring_area": int(ring.sum()),
        "vessels_area": int(vessels.sum()),
        "background_area": int(background.sum()),
        "bregma_found": bool(bregma_found),
        "bregma_cy": np.nan if cy is None else float(cy),
        "bregma_cx": np.nan if cx is None else float(cx),
    }

    if save_dir is not None:
        save_dir.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_dir / f"{stem}_mask_preview.png", dpi=150, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close(fig)
    return stats


preview_groups = _select_preview_groups(df_index)
max_stems = PREVIEW_MAX_STEMS_PER_GROUP if PREVIEW_MAX_STEMS_PER_GROUP is not None else int(_preview_cfg_value("max_stems_per_group", 6))
save_png = PREVIEW_SAVE_PNG if PREVIEW_SAVE_PNG is not None else bool(_preview_cfg_value("save_png", True))
show_inline = PREVIEW_SHOW_INLINE if PREVIEW_SHOW_INLINE is not None else bool(_preview_cfg_value("show_inline", True))
preview_dir = paths["step2_root"] / _preview_cfg_value("folder_name", "mask_previews") if save_png else None

preview_stats = []
for group_id in preview_groups:
    dfg = df_index[df_index["group_id"].astype(str).eq(str(group_id))].copy().head(max_stems)
    print(f"Preview group {group_id}: {len(dfg)} stem(s)")
    for _, row in dfg.iterrows():
        preview_stats.append(preview_step2_masks_for_row(row, save_dir=preview_dir, show=show_inline))

preview_stats_df = pd.DataFrame(preview_stats)
display(preview_stats_df)
if preview_dir is not None:
    preview_stats_df.to_csv(preview_dir / "mask_preview_stats.csv", index=False)
    print("Saved previews to:", preview_dir)


## 5. Inspect the reference selected for each registration group

The table below shows the selected `ref_stem`, the `group_id` and whether the row exactly matched `analysis_group.reference.match`. Selection criteria and any variant-key logic are read from YAML.

In [ ]:
rows = []
for group_id, dfg in df_index.groupby("group_id", sort=True):
    ref, exact = choose_reference_row_for_group(dfg, config=config, allow_fallback=True)
    row = {
        "group_id": group_id,
        "ref_stem": ref["stem"],
        "ref_exact_match": exact,
        "n_recordings": len(dfg),
    }
    for k in stem_keys:
        if k in ref.index:
            row[f"ref_{k}"] = ref[k]
    for c in dfg.columns:
        if c.endswith("_base") or c.endswith("_repeat"):
            row[f"ref_{c}"] = ref.get(c, "")
    rows.append(row)

ref_preview = pd.DataFrame(rows)
display(ref_preview)


In [ ]:
ref_preview.loc[ref_preview["ref_exact_match"] == False]

## 6. Optional notebook filters for testing

Use `GROUPS = None` for all eligible groups, or set `GROUPS` to a list of actual `group_id` values from your index. **Example only:** `GROUPS = ['m190']`.

`MANUAL_GROUPS_ONLY = True` requests a Step2 rerun limited to groups with saved manual bregma metadata. The code cell initializes this option to `False`.

In [ ]:
GROUPS = None
# Example group_id only: replace with a value from your own Step2 index.
# GROUPS = ["m190"]

MANUAL_GROUPS_ONLY = False
# MANUAL_GROUPS_ONLY = True

print("GROUPS:", GROUPS)
print("MANUAL_GROUPS_ONLY:", MANUAL_GROUPS_ONLY)


## 7. Run Step2 with progress bars

`run_step2(..., show_progress=True)` processes the configured recordings and writes masks and registration metadata. The `Step2 masks` progress bar iterates over stems; `Step2 registration` iterates over groups. This cell performs **real file writes**.

Review `step2.io` and the output paths before running. Registration estimates depend on the configured reference, ROI, feature method and bregma anchoring options.

In [ ]:
metadata_registration = run_step2(
    config_path=CONFIG_PATH,
    require_h5=True,
    show_progress=True,
    manual_groups_only=MANUAL_GROUPS_ONLY,
    groups=GROUPS,
)

print("rows:", len(metadata_registration))
print("saved:", paths["metadata_registration_csv"])
display(metadata_registration.head())


## 8. Registration and mask QC summary

Inspect `is_bad`, `bad_reasons`, registration scores, bregma status and saved mask paths in `metadata_registration.csv`.

For `step2.registration.mask_mode: group_union` or `hybrid`, a shared union mask can be written under the configured masks directory, for example:

```text
out/step2_masks/masks/_group_union/<group_id>/mask_for_reg_union.npy
```

**Important:** the actual registration ROI also depends on `step2.registration.roi_mode`. For `roi_mode: full_image`, Step2 uses the selected full-image ROI rather than necessarily using the shared union mask for scoring. Verify per-stem `mask_for_reg.npy` when reviewing the actual ROI.

In [ ]:
if not metadata_registration.empty:
    display(
        metadata_registration
        .groupby(["group_id", "is_bad"], dropna=False)
        .size()
        .reset_index(name="n")
    )

    bad = metadata_registration[metadata_registration["is_bad"].astype(bool)]
    if len(bad):
        cols = [
            "group_id", "stem", "ref_stem", "bad_reasons",
            "bregma_found", "bregma_source",
            "score", "mask_for_reg_area", "mask_for_reg_path",
        ]
        cols = [c for c in cols if c in bad.columns]
        display(bad[cols])
    else:
        print("No bad rows.")


## 9. Identify groups requiring manual bregma review

This is **Step2 bregma correction**, not the final anatomical landmarks of Step3. The standalone `step2_bregma_review.py` tool opens all stems in a selected group as a Napari stack. Verify or place **one bregma point for each stem/slice**; closing Napari allows the tool to save corrections to `metadata_registration.csv`.

Re-run Step2 registration for the affected groups with `--manual-groups-only`, then proceed to Step3.

In [ ]:
if paths["metadata_registration_csv"].exists():
    meta = pd.read_csv(paths["metadata_registration_csv"])
    missing_bregma = meta[
        (meta.get("bregma_found", pd.Series(False, index=meta.index)).astype(str).str.lower().isin(["false", "0", "no", "nan", ""]))
        | (meta.get("bad_reasons", pd.Series("", index=meta.index)).astype(str).str.contains("bregma_empty|ref_bregma_empty", regex=True, na=False))
    ].copy()

    if missing_bregma.empty:
        print("No missing bregma rows found.")
    else:
        display(missing_bregma[[c for c in ["group_id", "stem", "ref_stem", "bad_reasons", "bregma_found", "bregma_source"] if c in missing_bregma.columns]])
        print("Groups for step2_bregma_review.py:")
        print(sorted(missing_bregma["group_id"].astype(str).unique()))
else:
    print("metadata_registration.csv does not exist yet:", paths["metadata_registration_csv"])


## 10. Run interactive tools in a separate PowerShell terminal

Napari/Qt sessions and interactive Step3 console prompts are handled more reliably outside a JupyterLab cell. **Do not run the Step2/Step3 GUI commands through `subprocess.run()` inside this notebook.**

1. Open PowerShell in an environment with Napari and the required Python dependencies.
2. Use the `src` directory containing `step2_bregma_review.py`, `run_step3_all_groups.py` and the manual-registration widget.
3. Copy a command printed by the next cell. It uses the actual `SOURCE_DIR`, `CONFIG_PATH` and notebook Python interpreter.
4. After closing Napari, inspect the saved CSV tables and rerun the relevant step if required.

The next cell **only prints commands**; it does not launch a GUI. Any paths or `group_id` values shown in Markdown examples are placeholders and must be replaced. In particular, `m17` is **an example group ID**, not a default.

### 10a. Manual bregma review in Napari (PowerShell)

Review groups with missing bregma (default behavior):

```powershell
python .\step2_bregma_review.py --config "<PATH_TO_CONFIG.yaml>"
```

Review all stems in a particular group, including those with automatically detected bregma:

```powershell
python .\step2_bregma_review.py --config "<PATH_TO_CONFIG.yaml>" --groups <GROUP_ID> --all-selected
```

Replace the path and group placeholder with actual values. Manual coordinates are written to `metadata_registration.csv`; continue with the Step2 rerun below.

### 10b. Rerun registration after manual bregma correction

This is **not a GUI operation**. Either set `MANUAL_GROUPS_ONLY = True` in Section 7, or use PowerShell:

```powershell
python .\step2_masks_registration.py --config "<PATH_TO_CONFIG.yaml>" --manual-groups-only
```

To further restrict a rerun to a particular group:

```powershell
python .\step2_masks_registration.py --config "<PATH_TO_CONFIG.yaml>" --manual-groups-only --groups <GROUP_ID>
```

Review `metadata_registration.csv` after processing. Unselected stems should remain in the merged metadata when the output table already exists.

## 11. Step3: Napari QC and anatomical annotations (PowerShell)

`run_step3_all_groups.py` reads Step2 masks and registration metadata, displays group QC, allows manual corrections, and annotates anatomical landmarks on the selected reference stem. If `step3.registration_qc.ask_after_group` is enabled, it also asks for a console decision, so use a standalone terminal.

One group (replace the group placeholder):

```powershell
python .\run_step3_all_groups.py --config "<PATH_TO_CONFIG.yaml>" --groups <GROUP_ID>
```

All eligible groups:

```powershell
python .\run_step3_all_groups.py --config "<PATH_TO_CONFIG.yaml>"
```

The review and annotation settings come from `step3` in YAML unless explicitly overridden by CLI flags. If bregma corrections require a registration rerun, complete that Step2 rerun and repeat Step3 review before final manual registration.

## 12. Step3: final manual registration (PowerShell)

After Step3 registration QC, groups marked for manual registration in `registration_qc_groups.csv` can be inspected in the separate Napari widget:

```powershell
python .\step3_manual_registration_widget.py --config "<PATH_TO_CONFIG.yaml>" --bad-registration-only
```

To open one specified group (replace the example group ID):

```powershell
python .\step3_manual_registration_widget.py --config "<PATH_TO_CONFIG.yaml>" --groups <GROUP_ID>
```

Manual registration is saved separately from Step2 automatic transforms. Do not rerun the Step2 geometry after final manual approval without reviewing the saved override again.

## 13. Verify generated files

- `out/manifest_recordings.csv`: created by Step1; Step2/Step3 do not regenerate the manifest.
- `out/step2_masks/metadata_registration.csv`: maintained by Step2; subsequent manual bregma and Step3 corrections may update specific fields.
- `out/step2_masks/step2_missing_inputs.csv`: report of invalid or missing Step2 inputs, if any.
- `out/step3/step3_missing_inputs.csv`: Step3 records rejected for missing or invalid input masks, if any.
- `out/step3/frame_qc.csv`: manual frame QC; `reference_annotations.csv`: reference anatomical landmarks.
- `out/step3/registration_qc_groups.csv`: decisions on groups needing manual registration.
- `out/step3/manual_registration/manual_registration_overrides.csv`: final manual overrides, separate from Step2 auto registration.

These locations are examples using the default folder names. The corresponding directories and filenames may be changed in YAML.

In [ ]:
for key, path in [
    ("manifest", paths["manifest_path"]),
    ("Step2 metadata", paths["metadata_registration_csv"]),
    ("Step2 missing", paths["step2_root"] / "step2_missing_inputs.csv"),
    ("Step3 missing", paths["step3_root"] / "step3_missing_inputs.csv"),
    ("Step3 QC", paths["step3_root"] / "frame_qc.csv"),
    ("Step3 annotation", paths["step3_root"] / "reference_annotations.csv"),
    ("registration QC", paths["step3_root"] / "registration_qc_groups.csv"),
]:
    path = Path(path)
    print(f"{key:20} {'OK' if path.is_file() else 'missing'}  {path}")
